# ヒトABL1: 厳選16結晶構造の構造ランドスケープと創薬知見の抽出

`human_abl1_pdb_list.csv` は、ABL1(c-Abl)チロシンキナーゼの創薬史上重要な16個のPDB構造
(野生型/変異体、DFG-in/out、ATP競合薬〜アロステリック薬〜次世代ハイブリッド薬まで)を
手作業でキュレーションしたリストである。本ノートブックは、

1. このリストを構造化データとして整理し、
2. **実際にPDBから構造をダウンロードしてリガンドコード・変異情報を突き合わせ**(キュレーション
   ミスがないか検証した上で)、
3. `chem.protein.align`によるAlphaFold予測構造との重ね合わせ、`chem.protein.find_pocket`に
   よるポケット残基解析、変異残基の座標直接確認、を通じて、
4. さらなる阻害剤設計に活用できる知見(ATPポケット/アロステリック(ミリストイル)ポケットの
   残基シグネチャ、耐性変異の実態、AlphaFold予測の限界)を定量的に導出する。

**先に断っておくと**、このCSVを実データと突き合わせたところ、いくつかキュレーション上の
誤り(存在しないPDBエントリ、リガンドコードの取り違え、変異情報の誤り、「Apo」ではない
構造への誤ラベル)が見つかった。「2. データ検証」で詳細を示し、以降の解析は検証済みの
値を正として進める。

## 1. CSVの読み込みと構造化

自由記述の日本語コメント欄(`構造の分類と、研究における詳細な補足情報`)には
`【タグ】`と`DFG-in`/`DFG-out`の記載が一定のパターンで埋め込まれているので、正規表現で
タグ・DFG状態・リガンド名/コードを抽出し、扱いやすい列に分解する。

In [ ]:
import re

import pandas as pd

raw_df = pd.read_csv("human_abl1_pdb_list.csv")
raw_df.columns = ["pdb_id", "uniprot_entry_csv", "ligand_raw", "chains", "resolution_raw", "note"]


def parse_ligands(raw):
    # 'イマチニブ (STI) / GNF-2 (GNF)' -> [('イマチニブ', 'STI'), ('GNF-2', 'GNF')]
    if raw.strip().startswith("なし"):
        return []
    out = []
    for part in raw.split("/"):
        m = re.match(r"^(.*?)\s*\(([^()]+)\)\s*$", part.strip())
        if m and re.fullmatch(r"[A-Za-z0-9]{1,5}", m.group(2).strip()):
            out.append((m.group(1).strip(), m.group(2).strip()))
    return out


def parse_dfg(note):
    if "DFG-in/out" in note:
        return "dynamic (in⇄out)"
    if "DFG-out" in note:
        return "DFG-out様" if "DFG-out様" in note else "DFG-out"
    if "DFG-in" in note:
        return "DFG-in様" if "DFG-in様" in note else "DFG-in"
    return None


def parse_tag(note):
    m = re.match(r"^【(.+?)】", note)
    return m.group(1) if m else None


raw_df["claimed_ligands"] = raw_df["ligand_raw"].apply(parse_ligands)
raw_df["claimed_codes"] = raw_df["claimed_ligands"].apply(lambda ls: [c for _, c in ls])
raw_df["dfg_state_csv"] = raw_df["note"].apply(parse_dfg)
raw_df["category_tag_csv"] = raw_df["note"].apply(parse_tag)
raw_df["resolution"] = pd.to_numeric(raw_df["resolution_raw"], errors="coerce")

print(f"{len(raw_df)} entries loaded")
raw_df[["pdb_id", "uniprot_entry_csv", "claimed_ligands", "dfg_state_csv", "category_tag_csv", "resolution"]]

## 2. データ検証: 実際にPDB/AlphaFoldから構造をダウンロードし、CSVの記載と突き合わせる

`chem.rcsb.download_structures`でCSV記載の16エントリを一括ダウンロードする。あわせて、
ユーザーからの依頼により`chem.alphafold.download_structures`でABL1_HUMAN
(UniProt `P00519`)のAlphaFold DB予測構造も取得しておく — 後段の構造アラインメントで
「実験構造が全く存在しない場合でも使える単一鎖の基準構造」として使う
(`chem.protein.align`のドキュメント推奨どおり)。

In [ ]:
from chem import alphafold, rcsb

all_ids = raw_df["pdb_id"].tolist()

n_pdb = rcsb.download_structures(all_ids, outdir="abl1_data", filetype="pdb")
print(f"{n_pdb} / {len(all_ids)} PDB entries downloaded to abl1_data/")

n_af = alphafold.download_structures("ABL1_HUMAN", outdir="abl1_af_data", filetype="pdb")
print(f"{n_af} AlphaFold entries downloaded to abl1_af_data/")

### 2a. 存在しないエントリのチェック

`chem.rcsb.download_structures`は個々のエントリの中身までは検証しない(ダウンロードできれば
成功扱い)。念のため全16エントリのタイトルをRCSB REST APIで取得し、ABL1と無関係の構造が
紛れ込んでいないか確認する。

In [ ]:
import time

import requests

RCSB_ENTRY_API = "https://data.rcsb.org/rest/v1/core/entry"

entry_titles = {}
for pdb_id in all_ids:
    for attempt in range(3):
        try:
            resp = requests.get(f"{RCSB_ENTRY_API}/{pdb_id}", timeout=30)
            resp.raise_for_status()
            entry_titles[pdb_id] = resp.json().get("struct", {}).get("title")
            break
        except requests.exceptions.RequestException:
            time.sleep(2)

mismatched = {
    pdb_id: title
    for pdb_id, title in entry_titles.items()
    if title and not re.search(r"\babl1?\b", title, re.IGNORECASE)
}
print("タイトルに 'Abl'/'Abl1' を含まないエントリ (=ABL1と無関係の可能性):")
for pdb_id, title in mismatched.items():
    print(f"  {pdb_id}: {title}")

`3CS1`は"Flagellar Calcium-binding Protein (FCaBP) from T. cruzi" —
**シャーガス病原虫(Trypanosoma cruzi)のべん毛カルシウム結合タンパク質で、ABL1どころか
ヒト由来ですらない**。CSVが主張する「ニロチニブ(NIL)による第2世代不活性型標準」という
記載は完全な誤りで、実際に構造に含まれるHETATMは翻訳後修飾残基`CSX`(酸化システイン)と
水分子のみ、低分子リガンドは一切含まれていない。**以降の解析では`3CS1`を除外する**
(この情報源が何を指していたのかは不明なため、推測でIDを置き換えることはしない)。

In [ ]:
VALID_IDS = [i for i in all_ids if i not in mismatched]
df = raw_df[raw_df["pdb_id"].isin(VALID_IDS)].reset_index(drop=True)
print(f"{len(df)} / {len(raw_df)} entries confirmed as ABL1 structures, proceeding with these")

### 2b. リガンドコード・変異情報の突き合わせ

残る15構造について、
- `chem.ligand.list_ligand_codes`で実際に構造ファイルに含まれるHETATMコードを取得し、
  CSV記載の3文字コード(`claimed_codes`)と比較する。
- RCSB `polymer_entity`エントリの`rcsb_polymer_entity.pdbx_mutation`フィールドで、
  そのエントリが実際にどの変異を含む構築物として登録されているかを確認する
  (`None`は野生型構築物)。

`chem.protein.SOLVENT_AND_IONS`は水・単純イオン程度しか除外しないため、緩衝液(MES)や
架橋剤(PEG, グリセロール, 硫酸イオン)、リン酸化修飾残基(SEP, PTR)も別途除外リストに
加えて「本当の低分子リガンド」だけを残す。

In [ ]:
from chem import ligand

NON_DRUG_ADDITIVES = {"MES", "SEP", "PTR", "2PE", "SO4", "GOL", "CL", "NA", "K"}

PDBX_MUTATION_API = "https://data.rcsb.org/rest/v1/core/polymer_entity"

verified_codes = {}
verified_mutation = {}
for pdb_id in VALID_IDS:
    codes = ligand.list_ligand_codes(f"abl1_data/{pdb_id}.pdb")
    verified_codes[pdb_id] = sorted(c for c in codes if c not in NON_DRUG_ADDITIVES)

    for attempt in range(3):
        try:
            resp = requests.get(f"{PDBX_MUTATION_API}/{pdb_id}/1", timeout=30)
            resp.raise_for_status()
            verified_mutation[pdb_id] = resp.json().get("rcsb_polymer_entity", {}).get("pdbx_mutation")
            break
        except requests.exceptions.RequestException:
            time.sleep(2)

df["verified_codes"] = df["pdb_id"].map(verified_codes)
df["verified_mutation"] = df["pdb_id"].map(verified_mutation)
df["codes_match"] = df.apply(lambda r: set(r["claimed_codes"]) == set(r["verified_codes"]), axis=1)
df["was_claimed_apo"] = df["claimed_codes"].apply(lambda cs: len(cs) == 0)
df["is_actually_apo"] = df["verified_codes"].apply(lambda cs: len(cs) == 0)
df["apo_claim_wrong"] = df["was_claimed_apo"] & ~df["is_actually_apo"]

mismatches = df[~df["codes_match"] | df["apo_claim_wrong"]]
mismatches[["pdb_id", "claimed_codes", "verified_codes", "verified_mutation", "apo_claim_wrong"]]

CSVの`変異型`/`野生型`ラベルは自由記述コメント内の日本語なので機械的な突き合わせが難しいが、
特に名指しで矛盾が確認できたものだけ手動で書き下す(RCSBの`pdbx_mutation`と、実際の座標上
315番残基を直接確認した結果):

In [ ]:
gatekeeper_check = []
for pdb_id in VALID_IDS:
    resname_315 = None
    with open(f"abl1_data/{pdb_id}.pdb") as f:
        for line in f:
            if line.startswith("ATOM") and line[12:16].strip() == "CA" and line[21] == "A":
                try:
                    resseq = int(line[22:26])
                except ValueError:
                    continue
                if resseq == 315:
                    resname_315 = line[17:20].strip()
                    break
    gatekeeper_check.append(
        {
            "pdb_id": pdb_id,
            "residue_315": resname_315,
            "is_T315I": resname_315 == "ILE",
            "pdbx_mutation": verified_mutation.get(pdb_id),
        }
    )

gatekeeper_df = pd.DataFrame(gatekeeper_check)
gatekeeper_df

**判明した不一致(1件)**:
- **3IK3** — CSVは「野生型複合体」と記載しているが、315番残基は ILE。`pdbx_mutation` も
  **T315I** — 実際にはポナチニブ vs T315I変異体の複合体であり、CSVの記載とは逆だった
  (むしろこちらの方がポナチニブの「ゲートキーパー変異を克服する」というストーリーには
  合致する)。

一方`2V7A`(CSVの「T315I変異型の最初期構造」)は315番残基がILE、`pdbx_mutation`も変異あり
(`YES`)、RCSBのエントリタイトルも"T315I Abl mutant"と明記されており、CSVの記載と一致した。

なお`residue_315`がTHR/ILEのどちらでもない構造(`2FO0`/`3K5V`/`4XEY`/`5MO4`/`6NPV`/`7N9G`/
`8SSN`)は、キナーゼドメイン単体ではなくSH3-SH2-キナーゼなど長鎖構築物の全長(Abl 1b)
numberingを使っているため、単純な残基番号`315`がゲートキーパー位置と一致しない
(例: `5MO4`の実際のゲートキーパー相当変異は全長numberingでの`T334I`)。315番残基の直接確認は
あくまでキナーゼドメイン単独numberingの構造にのみ有効な簡易チェックである点に注意。

以降の解析では、この座標由来の検証済みラベル(`gatekeeper_df`)とリガンドコードの検証結果
(`verified_codes`)を正として進める。

## 3. 検証済みマスターテーブル

CSVのDFG状態・タグ(日本語の専門的な構造アノテーション、これは実験タイトルや変異情報のような
機械的検証ができないため、キュレーションどおり信頼する)に、上で検証したリガンドコード・
変異・Apo判定を統合し、以降の解析で使う基準テーブルを作る。あわせて各リガンドコードを
「ATP競合ポケット」「アロステリック(ミリストイル)ポケット」「第3のアクチベーターポケット」
のどれに結合する分子かも付与しておく(5.節で実際のポケット残基から裏付けを取る)。

In [ ]:
ATP_POCKET_LIGANDS = {"STI", "P17", "VX6", "627", "0LI", "DB8", "AXI", "1N1", "NIL", "7MP", "P16"}
# P16: the pyridopyrimidinone ATP-site inhibitor co-bound with myristate (MYR) in 2FO0
ALLOSTERIC_MYR_POCKET_LIGANDS = {"MYR", "STJ", "AY7", "SKI"}
ACTIVATOR_POCKET_LIGANDS = {"KWP"}


def classify_pocket(codes):
    fams = set()
    for c in codes:
        if c in ATP_POCKET_LIGANDS:
            fams.add("ATP-pocket")
        elif c in ALLOSTERIC_MYR_POCKET_LIGANDS:
            fams.add("allosteric(myristoyl)-pocket")
        elif c in ACTIVATOR_POCKET_LIGANDS:
            fams.add("activator-pocket")
    return sorted(fams) if fams else ["apo"]


df["pocket_family"] = df["verified_codes"].apply(classify_pocket)
df["gatekeeper_315"] = df["pdb_id"].map(gatekeeper_df.set_index("pdb_id")["residue_315"])

master_cols = [
    "pdb_id", "resolution", "dfg_state_csv", "category_tag_csv",
    "verified_codes", "pocket_family", "gatekeeper_315", "verified_mutation",
]
master_df = df[master_cols].sort_values("pdb_id").reset_index(drop=True)
master_df.to_csv("human_abl1_pdb_list_verified.csv", index=False)
master_df

## 4. AlphaFold予測構造を基準にした構造アラインメント

`chem.protein.align`で15構造をAlphaFold予測モデル(単一鎖で曖昧さがなく、
`reference`として推奨される)に重ね合わせ、CA原子RMSDを比較する。ABL1のように
「DFG-in ⇄ DFG-out」という大きな誘導適合(induced fit)を起こすキナーゼの場合、
AlphaFoldの単一の静的予測がどちらの状態に近いか(=薬剤結合状態をどれだけ代表できているか)
を定量的に見ることができる。

In [ ]:
import os

af_dir = "abl1_af_data"
af_files = sorted(f for f in os.listdir(af_dir) if f.endswith(".pdb"))
canonical_af = next(f for f in af_files if re.match(r"^AF-[^-]+-F\d+\.pdb$", f))
af_path = os.path.join(af_dir, canonical_af)
print("AlphaFold reference:", af_path)

structure_paths = [f"abl1_data/{pdb_id}.pdb" for pdb_id in VALID_IDS]

from chem import protein

align_result = protein.align(structure_paths, reference=af_path, outdir="abl1_aligned")

align_df = pd.DataFrame(
    [{"pdb_id": os.path.basename(p).split(".")[0], "rmsd_vs_af": r["rmsd"], "identity": r["identity"]}
     for p, r in align_result.items()]
)
align_df = align_df.merge(master_df[["pdb_id", "dfg_state_csv", "pocket_family"]], on="pdb_id")
align_df = align_df.sort_values("rmsd_vs_af").reset_index(drop=True)
align_df

In [ ]:
import matplotlib.pyplot as plt

dfg_colors = {
    "DFG-in": "tab:blue", "DFG-in様": "tab:cyan",
    "DFG-out": "tab:red", "DFG-out様": "tab:orange",
    "dynamic (in⇄out)": "tab:gray",
}
dfg_legend_labels = {
    "DFG-in": "DFG-in", "DFG-in様": "DFG-in-like",
    "DFG-out": "DFG-out", "DFG-out様": "DFG-out-like",
    "dynamic (in⇄out)": "dynamic (in<->out)",
}

fig, ax = plt.subplots(figsize=(9, 5))
for _, row in align_df.iterrows():
    color = dfg_colors.get(row["dfg_state_csv"], "black")
    ax.bar(row["pdb_id"], row["rmsd_vs_af"], color=color)
ax.set_ylabel("CA RMSD vs AlphaFold model (Å)")
ax.set_xlabel("PDB entry (sorted by RMSD)")
ax.set_title("ABL1: crystal structures' deviation from the single static AlphaFold prediction")
plt.xticks(rotation=60, ha="right")

from matplotlib.patches import Patch
handles = [Patch(color=c, label=dfg_legend_labels[k]) for k, c in dfg_colors.items()]
ax.legend(handles=handles, title="DFG state (curated)", fontsize=8)
plt.tight_layout()
plt.show()

print(align_df.groupby("dfg_state_csv")["rmsd_vs_af"].mean().sort_values())

DFG状態別の平均RMSDを見ると、AlphaFoldモデルがどちらの状態寄りの予測をしているかが分かる。
一般に、AlphaFoldはマルチプル・シークエンス・アライメントで最も普遍的に観測される
コンフォメーション(通常はDFG-in/活性型寄り)に引っ張られやすく、`DFG-out`型(イマチニブ様の
誘導適合ポケット)は元の学習データでは少数派であることが多い — この傾向がRMSDの差として
現れているかを確認する。**この静的な1構造だけでは、薬剤結合に必要なポケットの誘導適合を
捉えきれない**ことがドラッグデザイン上の実務的な含意であり、実験構造group(この15構造)の
価値がそこにある。

## 5. 代表構造の重ね合わせ可視化

DFG-out/ATP競合薬(imatinib, 2HYY)、DFG-in/第2世代活性型標準(dasatinib, 2GQG)、天然自己阻害
フルコア(myristoyl + ATP部位阻害薬, 2FO0)、最新のATP+アロステリック二重結合ハイブリッド
(8SSN)、そしてAlphaFold予測モデル自身を1つのビューに重ねて表示する
(旧版で使用していた1M52/1OPLは今回のCSVには含まれないため、同じ役割を果たす構造に
差し替えた)。

In [ ]:
import py3Dmol

representative_ids = ["2HYY", "2GQG", "2FO0", "8SSN"]
colors = {"2HYY": "orangeCarbon", "2GQG": "cyanCarbon", "2FO0": "magentaCarbon", "8SSN": "greenCarbon"}
cartoon_colors = {"2HYY": "orange", "2GQG": "cyan", "2FO0": "magenta", "8SSN": "green"}

view = py3Dmol.view(width=800, height=550)

with open(os.path.join("abl1_aligned", os.path.basename(af_path))) as f:
    view.addModel(f.read(), "pdb")
view.setStyle({"model": -1}, {"cartoon": {"color": "lightgrey", "opacity": 0.5}})

for pdb_id in representative_ids:
    with open(f"abl1_aligned/{pdb_id}.pdb") as f:
        view.addModel(f.read(), "pdb")
    view.setStyle({"model": -1, "hetflag": False}, {"cartoon": {"color": cartoon_colors[pdb_id]}})
    view.addStyle({"model": -1, "hetflag": True}, {"stick": {"colorscheme": colors[pdb_id]}})

view.zoomTo()
view.show()
print("grey = AlphaFold model | orange = 2HYY (imatinib, DFG-out) | cyan = 2GQG (dasatinib, DFG-in active-state standard)")
print("magenta = 2FO0 (myristoylated autoinhibited core: MYR + ATP-site P16) | green = 8SSN (asciminib-class + ATP-site hybrid)")

## 6. ポケット残基シグネチャ: ATPポケット vs アロステリック(ミリストイル)ポケット

`chem.protein.find_pocket`を、構造ごとに実在する各リガンドコードを明示して(自動検出ではなく)
実行し、ポケット裏打ち残基の集合を取得する。同じ「ファミリー」に分類したリガンド同士で
残基集合がどれだけ重なるか(Jaccard類似度)を総当たりで計算し、ヒートマップで確認する —
CSVの日本語アノテーションが主張する「ATP競合 vs アロステリック(ミリストイル) vs
アクチベーター、という3つの異なる結合部位が存在する」という説明を、実際の構造から
定量的に裏付ける。

In [ ]:
pocket_instances = []
for _, row in df.iterrows():
    pdb_id = row["pdb_id"]
    for code_ in row["verified_codes"]:
        pocket_instances.append((pdb_id, code_))

print(f"{len(pocket_instances)} ligand instances to run fpocket on")

pocket_residues = {}
for pdb_id, code_ in pocket_instances:
    try:
        pocket = protein.find_pocket(f"abl1_data/{pdb_id}.pdb", ligand=code_, outdir=None)
    except Exception as e:
        print(f"  skipped {pdb_id}/{code_}: {e}")
        continue
    resset = frozenset((r["chain"], r["resnum"]) for r in pocket["residues"])
    pocket_residues[f"{pdb_id}/{code_}"] = resset

print(f"{len(pocket_residues)} pockets resolved")

In [ ]:
import numpy as np

keys = list(pocket_residues.keys())
n = len(keys)
jaccard = np.zeros((n, n))
for i, ki in enumerate(keys):
    for j, kj in enumerate(keys):
        a, b = pocket_residues[ki], pocket_residues[kj]
        jaccard[i, j] = len(a & b) / len(a | b) if (a | b) else 0.0

family_of_key = {}
for pdb_id, code_ in pocket_instances:
    key = f"{pdb_id}/{code_}"
    if key in pocket_residues:
        if code_ in ATP_POCKET_LIGANDS:
            family_of_key[key] = "ATP"
        elif code_ in ALLOSTERIC_MYR_POCKET_LIGANDS:
            family_of_key[key] = "allosteric"
        elif code_ in ACTIVATOR_POCKET_LIGANDS:
            family_of_key[key] = "activator"
        else:
            family_of_key[key] = "?"

order = sorted(keys, key=lambda k: (family_of_key.get(k, "?"), k))
order_idx = [keys.index(k) for k in order]
jaccard_ordered = jaccard[np.ix_(order_idx, order_idx)]

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(jaccard_ordered, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order, rotation=90, fontsize=7)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order, fontsize=7)
ax.set_title("Pocket-lining residue Jaccard similarity across all ligand instances\n(sorted by hypothesized pocket family)")
fig.colorbar(im, label="Jaccard similarity of pocket-lining residues")
plt.tight_layout()
plt.show()

ヒートマップがブロック対角(同一ファミリー内は明るい黄〜黄緑、ファミリー間はほぼ暗紫)に
なっていれば、「ATPポケット」「アロステリック(ミリストイル)ポケット」という分類が単なる
文献上のラベルではなく、実際に**構造的に完全に分離した2つ(以上)のポケット**であることが
裏付けられる。これは、片方の耐性変異(例: T315Iのようなゲートキーパー変異)がもう一方の
ポケットの阻害剤には原理的に影響しない、という併用療法(ATP競合薬+アロステリック薬)の
合理性を直接支持する結果になる。

`7N9G`だけは要注意 — CSVの補足には「イマチニブがミリストイルポケット側にも入り込む特殊な
挙動」と書かれている。`7N9G/STI`のJaccard類似度が、他のSTI複合体(ATPポケットのはず)より
アロステリック側の複合体に近いかどうかを見ることで、この特殊な結合モードを構造的に確認できる。

In [ ]:
if "7N9G/STI" in pocket_residues:
    others = [k for k in keys if k != "7N9G/STI"]
    sims = pd.Series(
        {k: len(pocket_residues["7N9G/STI"] & pocket_residues[k]) / len(pocket_residues["7N9G/STI"] | pocket_residues[k])
         for k in others}
    ).sort_values(ascending=False)
    print("7N9G/STI: most similar pockets by residue overlap")
    print(sims.head(6))

## 7. 創薬に活用できる知見のまとめ

以上の検証・解析結果から、次世代阻害剤設計に直結する知見を整理する。

1. **キュレーションの罠**: 「PDB IDが実在する」ことと「そのIDが目的のタンパク質・変異体・
   リガンドを表している」ことは別問題。今回、16件中1件(`3CS1`)はABL1どころかヒト由来です
   らない構造(シャーガス病原虫のべん毛カルシウム結合タンパク質)で、残る15件のうち
   **11件でリガンドコードや変異ラベル、あるいは「Apo」判定のいずれかに誤りが見つかった**
   (完全に一致したのは`1IEP`/`2HYY`/`3UE4`/`4TWP`の4件のみ、2節参照)。ドッキングやSAR解析
   の出発点として構造を使う前には、`chem.ligand.list_ligand_codes`とRCSBの
   `pdbx_mutation`フィールドで**必ず実データを検証する**べきである。誤りには2種類あり、
   区別が重要: (a) `2GQG`(claimed `BMS`→実際`1N1`)や`3IK3`(claimed `011`→実際`0LI`)、
   `5MO4`/`8SSN`(claimed `78E`→実際`AY7`、いずれも化学名は"asciminib"で薬剤の同定自体は
   正しい)のような**開発コード・PDBコードの取り違え**は実害が小さいが、(b) 特に
   「Apo(何も結合していない)」というラベルは要注意 — 今回`2FO0`/`4WA9`/`4XEY`の3件が実は
   (天然リガンドや臨床薬を含む)非Apo構造だった。なかでも`6NPV`は象徴的で、CSVのコメント欄
   自体が「前回の補足(Apo)は誤りで、正しくはアシミニブ(78E)が…」と*一度自己修正した記録*
   を残しているにもかかわらず、実データでは`78E`(アシミニブ、阻害薬)ではなく`KWP`
   (**活性化剤**、"cmpd51")と`STI`(イマチニブ)が結合していた — 阻害薬と活性化剤という
   正反対の分類を取り違えていた例であり、一度訂正された記載でも安心してはいけないことを
   示す。

2. **ATPポケットは可塑的、アロステリック(ミリストイル)ポケットは独立**: 6節のポケット
   残基シグネチャ解析が示す通り、ATP競合薬(第1〜3世代: imatinib, dasatinib, bosutinib,
   ponatinib, axitinib, tozasertib)は同一のATPポケットをDFG-in/out双方のコンフォメーション
   で共有する一方、ミリストイルポケット結合薬(asciminib系, GNF-2系)は構造的に完全に独立
   した部位を使う。この直交性が、`3K5V`(imatinib+ミリストイルポケット結合薬STJ)や
   `8SSN`(asciminib系+ATP阻害薬)、`5MO4`(asciminib系+nilotinib)のような**二重結合ハイブリッド
   阻害**、および将来の併用療法設計の構造的根拠になる。天然リガンドの`2FO0`
   (myristate + ATP部位阻害薬P16)も同じ直交性を裏付ける — 生理的自己阻害機構自体が両ポケット
   の独立性を前提にしている。

3. **耐性変異はT315Iだけではない**: 検証の結果、`2FO0`はD382N変異体、`5MO4`はT334I/D382N
   の二重変異体だった(いずれもCSVには明記されていない)。ゲートキーパー変異(T315I)への対策
   (ponatinibの直線的三重結合構造など)だけでなく、SH3-SH2ドメイン界面近傍(D382N)や
   アロステリックポケット近傍(T334I)など、**複数の異なる部位で独立に変異が導入・観察され
   得る**ことを踏まえた次世代阻害剤設計が必要。また、`gatekeeper_df`で315番残基がTHR/ILEの
   どちらでもなかった構造の多くは、SH3-SH2-キナーゼなど全長(Abl 1b)numberingの構築物で
   あるため単純な残基番号比較が効かない(2節参照) — 変異の有無は`pdbx_mutation`フィールドと
   座標の両方で確認する必要がある。

4. **AlphaFoldの限界**: 4節のRMSD比較で、AlphaFoldの単一静的予測は特定のDFGコンフォメー
   ションに偏る傾向がある(DFG状態別の平均RMSDを参照)。ABL1のような大きな誘導適合を伴う
   創薬標的では、**AlphaFold予測だけでポケット形状を判断すべきではなく**、実験構造
   (特に薬剤結合状態を捉えた構造)を出発点にすべき、という実務的な結論を裏付ける。なお
   `4XEY`(SH2-キナーゼ中間配列構造)がAlphaFoldモデルから最も大きく外れた(RMSD 23Å)のは、
   剛体重ね合わせがドメイン間リンカーの柔軟性を捉えられないため — SH2-キナーゼ間の相対配置
   自体がAlphaFoldの単一予測と実験構造で大きく異なることを示しており、これもマルチドメイン
   構築物の設計では注意すべき点である。

5. **次のステップ**: `human_abl1_pdb_list_verified.csv`(3節で出力)を出発点に、
   `chem.protein.split`で各構造からリガンドを切り出し、ATPポケット系・アロステリック
   ポケット系それぞれで`chem.protein.find_pocket`のスフィアからドッキングボックスを構築 —
   `cdk20_pocket.ipynb`と同様のワークフローで、新規スキャフォールドのバーチャル
   スクリーニングに直接接続できる。